# Machine Learning

In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split, RandomizedSearchCV, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
import warnings
from scipy.stats import uniform, randint
import sys
import timm
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, random_split, DataLoader, Subset, ConcatDataset
import copy

c:\Users\14793\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_parquet("../data/processed/anime_data_2.parquet")
df.info()

<class 'pandas.DataFrame'>
Index: 5338 entries, 0 to 8816
Data columns (total 76 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   mal_id                 5338 non-null   int64  
 1   title                  5338 non-null   str    
 2   source                 5338 non-null   str    
 3   episodes               5338 non-null   float64
 4   producers              5338 non-null   object 
 5   genres                 5338 non-null   object 
 6   studios                5338 non-null   object 
 7   demographics           5338 non-null   object 
 8   themes                 5338 non-null   object 
 9   rating                 5338 non-null   str    
 10  sequel                 5338 non-null   bool   
 11  cohort                 5338 non-null   str    
 12  wc_z                   5338 non-null   float64
 13  forum_z                5338 non-null   float64
 14  favorites_z            5338 non-null   float64
 15  score_z             

## Input Preparation

Right now, the priority is to reduce dimensions. The plan is the following:
* Reduce the dimensions of the image tensors from 512
* Reduce the dimensions of sentimental analysis tensors from 768
* Reduce the pool of producers and studios into embedded vectors

In [3]:
df.info()

<class 'pandas.DataFrame'>
Index: 5338 entries, 0 to 8816
Data columns (total 76 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   mal_id                 5338 non-null   int64  
 1   title                  5338 non-null   str    
 2   source                 5338 non-null   str    
 3   episodes               5338 non-null   float64
 4   producers              5338 non-null   object 
 5   genres                 5338 non-null   object 
 6   studios                5338 non-null   object 
 7   demographics           5338 non-null   object 
 8   themes                 5338 non-null   object 
 9   rating                 5338 non-null   str    
 10  sequel                 5338 non-null   bool   
 11  cohort                 5338 non-null   str    
 12  wc_z                   5338 non-null   float64
 13  forum_z                5338 non-null   float64
 14  favorites_z            5338 non-null   float64
 15  score_z             

In [4]:
all_indices = np.arange(len(df))

train_idx, test_idx = train_test_split(
    all_indices,
    test_size=0.10,
    random_state=456
)

train_idx, val_idx = train_test_split(
    train_idx,
    test_size=0.10,
    random_state=456
)

print("Train:", len(train_idx))
print("Validation:", len(val_idx))
print("Test:", len(test_idx))

Train: 4323
Validation: 481
Test: 534


In [5]:
studio_to_idx = {"<UNK>": 0}
producer_to_idx = {"<UNK>": 0}

for studios in df.iloc[train_idx]["studios"]:
    for studio in studios:
        if studio not in studio_to_idx:
            studio_to_idx[studio] = len(studio_to_idx)

for producers in df.iloc[train_idx]["producers"]:
    for producer in producers:
        if producer not in producer_to_idx:
            producer_to_idx[producer] = len(producer_to_idx)

n_studios = len(studio_to_idx)
n_producers = len(producer_to_idx)

print("Number of studios:", n_studios)
print("Number of producers:", n_producers)

Number of studios: 481
Number of producers: 1013


In [6]:
def get_studio_indices(studios):
    return [
        studio_to_idx.get(studio, 0)
        for studio in studios
    ]

def get_producer_indices(producers):
    return [
        producer_to_idx.get(producer, 0)
        for producer in producers
    ]

df["studio_idx"] = df["studios"].apply(get_studio_indices)
df["producer_idx"] = df["producers"].apply(get_producer_indices)

print(df["studio_idx"].head())
print(df["producer_idx"].head())

0     [14]
1     [20]
2     [14]
3     [19]
4    [139]
Name: studio_idx, dtype: object
0       [117, 164, 711]
1             [164, 47]
2        [117, 99, 164]
3              [13, 99]
4    [13, 159, 162, 53]
Name: producer_idx, dtype: object


In [7]:
def create_embedding_bag_inputs(index_lists):
    flat_indices = []
    offsets = []

    current_offset = 0

    for indices in index_lists:
        offsets.append(current_offset)
        flat_indices.extend(indices)
        current_offset += len(indices)

    return (
        torch.tensor(flat_indices, dtype=torch.long),
        torch.tensor(offsets, dtype=torch.long)
    )


studio_indices, studio_offsets = create_embedding_bag_inputs(
    df["studio_idx"]
)

producer_indices, producer_offsets = create_embedding_bag_inputs(
    df["producer_idx"]
)

print("Studio indices:", studio_indices.shape)
print("Studio offsets:", studio_offsets.shape)

print("Producer indices:", producer_indices.shape)
print("Producer offsets:", producer_offsets.shape)

Studio indices: torch.Size([5725])
Studio offsets: torch.Size([5338])
Producer indices: torch.Size([18230])
Producer offsets: torch.Size([5338])


In [8]:
def split_embedding_bag_inputs(index_lists, train_idx, val_idx, test_idx):
    
    def create_for_rows(rows):
        selected_lists = [index_lists[i] for i in rows]
        return create_embedding_bag_inputs(selected_lists)

    train_indices, train_offsets = create_for_rows(train_idx)
    val_indices, val_offsets = create_for_rows(val_idx)
    test_indices, test_offsets = create_for_rows(test_idx)

    return (
        train_indices, train_offsets,
        val_indices, val_offsets,
        test_indices, test_offsets
    )


(
    studio_indices_train,
    studio_offsets_train,
    studio_indices_val,
    studio_offsets_val,
    studio_indices_test,
    studio_offsets_test
) = split_embedding_bag_inputs(
    df["studio_idx"].tolist(),
    train_idx,
    val_idx,
    test_idx
)


(
    producer_indices_train,
    producer_offsets_train,
    producer_indices_val,
    producer_offsets_val,
    producer_indices_test,
    producer_offsets_test
) = split_embedding_bag_inputs(
    df["producer_idx"].tolist(),
    train_idx,
    val_idx,
    test_idx
)

### Synopsis

In [9]:
semantic_embeddings = np.load('../data/processed/semantic_embeddings.npy')

In [10]:
text_scaler = StandardScaler()

text_scaler.fit(
    semantic_embeddings[train_idx]
)

semantic_embeddings_train = text_scaler.transform(
    semantic_embeddings[train_idx]
)

semantic_embeddings_val = text_scaler.transform(
    semantic_embeddings[val_idx]
)

semantic_embeddings_test = text_scaler.transform(
    semantic_embeddings[test_idx]
)

print(
    "Train text NaNs:",
    np.isnan(semantic_embeddings_train).sum()
)

print(
    "Val text NaNs:",
    np.isnan(semantic_embeddings_val).sum()
)

print(
    "Test text NaNs:",
    np.isnan(semantic_embeddings_test).sum()
)

print(
    "Train text infs:",
    np.isinf(semantic_embeddings_train).sum()
)

Train text NaNs: 0
Val text NaNs: 0
Test text NaNs: 0
Train text infs: 0


In [11]:
class LearnedProjector(nn.Module):
    """
    Projects a frozen all-mpnet-base-v2 sentence embedding (768-dim)
    down to a smaller learned representation via a linear layer.

    Note: sentence-transformers' mpnet output is L2-normalized by default
    (normalize_embeddings=True), so no extra normalization is applied here
    on the input side.

    Usage:
        projector = LearnedProjector(in_dim=768, out_dim=64)
        z = projector(x)  # x: (batch, 768) -> z: (batch, 64)
    """
    def __init__(self, in_dim: int = 768, out_dim: int = 64,
                 hidden_dim: int | None = None, dropout: float = 0.1):
        super().__init__()

        if hidden_dim is None:
            self.net = nn.Sequential(
                nn.Linear(in_dim, out_dim),
                nn.LayerNorm(out_dim)
            )
        else:
            self.net = nn.Sequential(
                nn.Linear(in_dim, hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, out_dim),
                nn.LayerNorm(out_dim)
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

projector = LearnedProjector(in_dim=768, out_dim=64)

## Images

In [12]:
image_data = np.load("../data/processed/compressed_image_embeddings.npy")

In [13]:
image_scaler = StandardScaler()

image_scaler.fit(
    image_data[train_idx]
)

image_train = image_scaler.transform(
    image_data[train_idx]
)

image_val = image_scaler.transform(
    image_data[val_idx]
)

image_test = image_scaler.transform(
    image_data[test_idx]
)

print("Image train NaNs:", np.isnan(image_train).sum())
print("Image val NaNs:", np.isnan(image_val).sum())
print("Image test NaNs:", np.isnan(image_test).sum())

Image train NaNs: 0
Image val NaNs: 0
Image test NaNs: 0


## Real Data Preparation

We've loaded real data from seasonal_anime.py.

## Fusion Network

### Score Prediction

Before making the network, let's confirm dimensions in the "other" category since it's unclear.

In [14]:
df.info()

<class 'pandas.DataFrame'>
Index: 5338 entries, 0 to 8816
Data columns (total 78 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   mal_id                 5338 non-null   int64  
 1   title                  5338 non-null   str    
 2   source                 5338 non-null   str    
 3   episodes               5338 non-null   float64
 4   producers              5338 non-null   object 
 5   genres                 5338 non-null   object 
 6   studios                5338 non-null   object 
 7   demographics           5338 non-null   object 
 8   themes                 5338 non-null   object 
 9   rating                 5338 non-null   str    
 10  sequel                 5338 non-null   bool   
 11  cohort                 5338 non-null   str    
 12  wc_z                   5338 non-null   float64
 13  forum_z                5338 non-null   float64
 14  favorites_z            5338 non-null   float64
 15  score_z             

In [15]:
X_other_pre = df.drop(columns=['studio_idx', 'producer_idx', 'mal_id', 'episodes', 'title', 'source', 'cohort', 'producers', 'rating', 'genres', 'studios', 'demographics', 'themes', 'score_z', 'wc_z', 'favorites_z', 'drop_rate_z', 'forum_z'])
X_other_pre['sequel'] = X_other_pre['sequel'].astype(int)
X_other_pre = X_other_pre.reset_index(drop=True)
X_other_pre.info()

<class 'pandas.DataFrame'>
RangeIndex: 5338 entries, 0 to 5337
Data columns (total 60 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   sequel                 5338 non-null   int64
 1   genre_action           5338 non-null   int64
 2   genre_adventure        5338 non-null   int64
 3   genre_award_winning    5338 non-null   int64
 4   genre_comedy           5338 non-null   int64
 5   genre_drama            5338 non-null   int64
 6   genre_ecchi            5338 non-null   int64
 7   genre_fantasy          5338 non-null   int64
 8   genre_gourmet          5338 non-null   int64
 9   genre_horror           5338 non-null   int64
 10  genre_mystery          5338 non-null   int64
 11  genre_other            5338 non-null   int64
 12  genre_romance          5338 non-null   int64
 13  genre_sci-fi           5338 non-null   int64
 14  genre_slice_of_life    5338 non-null   int64
 15  genre_sports           5338 non-null   int64
 16 

In [16]:
adaptation_df = pd.read_parquet("../data/processed/adaptation_data.parquet")

In [17]:
adaptation_df['has_adaptation_score'] = adaptation_df['adaptation_score'].notna().astype(int)
adaptation_df['has_adaptation_members'] = adaptation_df['adaptation_members'].notna().astype(int)

In [18]:
other_scaler = StandardScaler()

features = ['adaptation_score', 'adaptation_members']

print(adaptation_df[features].isna().sum())

train_data = adaptation_df.iloc[train_idx][features]
val_data   = adaptation_df.iloc[val_idx][features]
test_data  = adaptation_df.iloc[test_idx][features]

imputer = SimpleImputer(strategy='mean') # or 'median'
train_imputed = imputer.fit_transform(train_data)
val_imputed   = imputer.transform(val_data)
test_imputed  = imputer.transform(test_data)

adaptation_train = other_scaler.fit_transform(train_imputed)
adaptation_val   = other_scaler.transform(val_imputed)
adaptation_test  = other_scaler.transform(test_imputed)

adaptation_score      2670
adaptation_members    1996
dtype: int64


In [19]:
adaptation_df.loc[train_idx, 'adaptation_score'] = adaptation_train[:, 0]
adaptation_df.loc[val_idx, 'adaptation_score'] = adaptation_val[:, 0]
adaptation_df.loc[test_idx, 'adaptation_score'] = adaptation_test[:, 0]

adaptation_df.loc[train_idx, 'adaptation_members'] = adaptation_train[:, 1]
adaptation_df.loc[val_idx, 'adaptation_members'] = adaptation_val[:, 1]
adaptation_df.loc[test_idx, 'adaptation_members'] = adaptation_test[:, 1]

adaptation_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5338 entries, 0 to 5337
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   adaptation_score        5338 non-null   float64
 1   adaptation_members      5338 non-null   float64
 2   has_adaptation_score    5338 non-null   int64  
 3   has_adaptation_members  5338 non-null   int64  
dtypes: float64(2), int64(2)
memory usage: 166.9 KB


In [20]:
class FusionNetwork(nn.Module):

    def __init__(self):
        super().__init__()

        # Text projector
        self.text_projector = LearnedProjector(
            in_dim=768,
            hidden_dim=128,
            out_dim=64,
            dropout=0.4
        )

        # Image branch
        self.image_branch = nn.Sequential(
            nn.Linear(417, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.4)
        )

        # Tabular branch
        self.other_branch = nn.Sequential(
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.4)
        )

        self.studio_embedding = nn.EmbeddingBag(
            num_embeddings=n_studios,
            embedding_dim=8,
            mode='mean'
        )

        self.producer_embedding = nn.EmbeddingBag(
            num_embeddings=n_producers,
            embedding_dim=8,
            mode='mean'
        )

        # Fusion
        self.fusion = nn.Sequential(
            nn.Linear(64 + 64 + 32 + 8 + 8, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

    def forward(self, text, image, other, studio_indices, studio_offsets, producer_indices, producer_offsets):
        # 1. Check raw inputs
        for name, tensor in [('text', text), ('image', image), ('other', other)]:
            if torch.isnan(tensor).any():
                print(f"NaN detected in raw input: {name}")

        text_features = self.text_projector(text)
        image_features = self.image_branch(image)
        other_features = self.other_branch(other)
        
        # 2. Check branches
        if torch.isnan(text_features).any(): print("NaN in text branch")
        if torch.isnan(image_features).any(): print("NaN in image branch (Check BatchNorm/Batch Size)")
        if torch.isnan(other_features).any(): print("NaN in other branch")

        studio_features = self.studio_embedding(studio_indices, studio_offsets)
        producer_features = self.producer_embedding(producer_indices, producer_offsets)
        
        # 3. Check embeddings
        if torch.isnan(studio_features).any(): print("NaN in studio embeddings")

        combined = torch.cat([text_features, image_features, other_features, studio_features, producer_features], dim=1)
        
        out = self.fusion(combined)
        if torch.isnan(out).any(): print("NaN generated inside Fusion layers")
            
        return out

In [293]:
X_text_train = torch.from_numpy(
    semantic_embeddings_train.astype(np.float32)
)

X_text_val = torch.from_numpy(
    semantic_embeddings_val.astype(np.float32)
)

X_text_test = torch.from_numpy(
    semantic_embeddings_test.astype(np.float32)
)

X_image_train = torch.from_numpy(
    image_train.astype(np.float32)
)

X_image_val = torch.from_numpy(
    image_val.astype(np.float32)
)

X_image_test = torch.from_numpy(
    image_test.astype(np.float32)
)

temp_merge = pd.concat(
    [X_other_pre, adaptation_df],
    axis=1
)

X_other = torch.from_numpy(
    temp_merge.to_numpy(dtype=np.float32)
)

y_score = torch.tensor(
    df["score_z"].to_numpy(dtype=np.float32)
)

has_nan = torch.isnan(X_text).any().item()
has_nan

False

In [294]:
other_train = X_other[train_idx]
other_val = X_other[val_idx]
other_test = X_other[test_idx]

score_train = y_score[train_idx]
score_val = y_score[val_idx]
score_test = y_score[test_idx]

In [26]:
model = FusionNetwork()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)


class AnimeDataset(torch.utils.data.Dataset):

    def __init__(
        self,
        text,
        image,
        other,
        target,
        studio_indices,
        studio_offsets,
        producer_indices,
        producer_offsets
    ):
        self.text = text
        self.image = image
        self.other = other
        self.target = target

        self.studio_indices = studio_indices
        self.studio_offsets = studio_offsets

        self.producer_indices = producer_indices
        self.producer_offsets = producer_offsets

    def __len__(self):
        return len(self.target)

    def __getitem__(self, idx):

        # Figure out where this anime's studio list starts
        studio_start = self.studio_offsets[idx]

        if idx + 1 < len(self.studio_offsets):
            studio_end = self.studio_offsets[idx + 1]
        else:
            studio_end = len(self.studio_indices)

        studio_indices = self.studio_indices[
            studio_start:studio_end
        ]

        # Same thing for producers
        producer_start = self.producer_offsets[idx]

        if idx + 1 < len(self.producer_offsets):
            producer_end = self.producer_offsets[idx + 1]
        else:
            producer_end = len(self.producer_indices)

        producer_indices = self.producer_indices[
            producer_start:producer_end
        ]

        return (
            self.text[idx],
            self.image[idx],
            self.other[idx],
            studio_indices,
            producer_indices,
            self.target[idx]
        )

def collate_fn(batch):

    texts = torch.stack([item[0] for item in batch])
    images = torch.stack([item[1] for item in batch])
    others = torch.stack([item[2] for item in batch])
    targets = torch.stack([item[5] for item in batch])

    studio_indices = []
    studio_offsets = []

    current_offset = 0

    for item in batch:
        indices = item[3]

        studio_offsets.append(current_offset)

        studio_indices.extend(
            indices.tolist()
        )

        current_offset += len(indices)

    studio_indices = torch.tensor(
        studio_indices,
        dtype=torch.long
    )

    studio_offsets = torch.tensor(
        studio_offsets,
        dtype=torch.long
    )

    producer_indices = []
    producer_offsets = []

    current_offset = 0

    for item in batch:
        indices = item[4]

        producer_offsets.append(current_offset)

        producer_indices.extend(
            indices.tolist()
        )

        current_offset += len(indices)

    producer_indices = torch.tensor(
        producer_indices,
        dtype=torch.long
    )

    producer_offsets = torch.tensor(
        producer_offsets,
        dtype=torch.long
    )

    return (
        texts,
        images,
        others,
        studio_indices,
        studio_offsets,
        producer_indices,
        producer_offsets,
        targets
    )

In [296]:
train_dataset = AnimeDataset(
    X_text_train,
    X_image_train,
    other_train,
    score_train,
    studio_indices_train,
    studio_offsets_train,
    producer_indices_train,
    producer_offsets_train
)

val_dataset = AnimeDataset(
    X_text_val,
    X_image_val,
    other_val,
    score_val,
    studio_indices_val,
    studio_offsets_val,
    producer_indices_val,
    producer_offsets_val
)

test_dataset = AnimeDataset(
    X_text_test,
    X_image_test,
    other_test,
    score_test,
    studio_indices_test,
    studio_offsets_test,
    producer_indices_test,
    producer_offsets_test
)

In [297]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
)


In [298]:
full_dataset = ConcatDataset([train_dataset, val_dataset])
n_samples = len(full_dataset)

k = 5
kfold = KFold(n_splits=k, shuffle=True, random_state=42)

batch_size = train_loader.batch_size
collate_fn = train_loader.collate_fn 

criterion = nn.MSELoss()
patience = 15
n_epochs = 100

fold_results = []          
fold_model_states = []    

for fold, (train_idx, val_idx) in enumerate(kfold.split(np.arange(n_samples))):

    print(f"\n===== Fold {fold + 1}/{k} =====")

    model = FusionNetwork().to(device)  
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=2e-4,
        weight_decay=1e-2
    )

    train_subset = Subset(full_dataset, train_idx)
    val_subset = Subset(full_dataset, val_idx)

    fold_train_loader = DataLoader(
        train_subset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_fn
    )
    fold_val_loader = DataLoader(
        val_subset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn
    )

    best_val_loss = float("inf")
    patience_counter = 0
    best_model_state = None

    for epoch in range(n_epochs):

        model.train()
        train_loss = 0

        for (
            text, image, other,
            studio_indices, studio_offsets,
            producer_indices, producer_offsets,
            target
        ) in fold_train_loader:

            text = text.to(device)
            image = image.to(device)
            other = other.to(device)
            studio_indices = studio_indices.to(device)
            studio_offsets = studio_offsets.to(device)
            producer_indices = producer_indices.to(device)
            producer_offsets = producer_offsets.to(device)
            target = target.to(device)

            prediction = model(
                text, image, other,
                studio_indices, studio_offsets,
                producer_indices, producer_offsets
            )

            loss = criterion(prediction.squeeze(-1), target)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        avg_train_loss = train_loss / len(fold_train_loader)

        model.eval()
        val_loss = 0

        with torch.no_grad():
            for (
                text, image, other,
                studio_indices, studio_offsets,
                producer_indices, producer_offsets,
                target
            ) in fold_val_loader:

                text = text.to(device)
                image = image.to(device)
                other = other.to(device)
                studio_indices = studio_indices.to(device)
                studio_offsets = studio_offsets.to(device)
                producer_indices = producer_indices.to(device)
                producer_offsets = producer_offsets.to(device)
                target = target.to(device)

                prediction = model(
                    text, image, other,
                    studio_indices, studio_offsets,
                    producer_indices, producer_offsets
                )

                loss = criterion(prediction.squeeze(-1), target)
                val_loss += loss.item()

        avg_val_loss = val_loss / len(fold_val_loader)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            best_model_state = copy.deepcopy(model.state_dict())
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Fold {fold + 1} early stopping at epoch {epoch}.")
                break

        if epoch % 5 == 0 or patience_counter == 0:
            print(
                f"  Epoch {epoch}: "
                f"Train Loss: {avg_train_loss:.4f} | "
                f"Val Loss: {avg_val_loss:.4f}"
            )

    print(f"Fold {fold + 1} best val loss: {best_val_loss:.4f}")
    fold_results.append(best_val_loss)
    fold_model_states.append(best_model_state)

fold_results = np.array(fold_results)
print(f"\n===== CV Results ({k}-fold) =====")
print(f"Per-fold val loss: {fold_results}")
print(f"Mean: {fold_results.mean():.4f}  |  Std: {fold_results.std():.4f}")

best_fold_idx = fold_results.argmin()
best_model_state = fold_model_states[best_fold_idx]
model = FusionNetwork().to(device)
model.load_state_dict(best_model_state)
print(f"\nLoaded weights from fold {best_fold_idx + 1} (val loss {fold_results[best_fold_idx]:.4f})")


===== Fold 1/5 =====


KeyboardInterrupt: 

In [ ]:
model.eval()

total_squared_error = 0
total_samples = 0

all_predictions = []
all_targets = []

with torch.no_grad():

    for (
        text,
        image,
        other,
        studio_indices,
        studio_offsets,
        producer_indices,
        producer_offsets,
        target
    ) in test_loader:

        text = text.to(device)
        image = image.to(device)
        other = other.to(device)

        studio_indices = studio_indices.to(device)
        studio_offsets = studio_offsets.to(device)

        producer_indices = producer_indices.to(device)
        producer_offsets = producer_offsets.to(device)

        target = target.to(device)

        predictions = model(
            text,
            image,
            other,
            studio_indices,
            studio_offsets,
            producer_indices,
            producer_offsets
        ).squeeze(-1)

        squared_error = (
            (predictions - target) ** 2
        ).sum()

        total_squared_error += squared_error.item()
        total_samples += target.size(0)

        all_predictions.append(
            predictions.cpu()
        )

        all_targets.append(
            target.cpu()
        )

final_mse = (
    total_squared_error /
    total_samples
)

final_rmse = np.sqrt(final_mse)

final_predictions = torch.cat(
    all_predictions
).numpy()

final_targets = torch.cat(
    all_targets
).numpy()

print(f"Test MSE:  {final_mse:.4f}")
print(f"Test RMSE: {final_rmse:.4f}")

Test MSE:  1.0256
Test RMSE: 1.0127


### WC Prediction

In [64]:
model = FusionNetwork()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

X_text_train = torch.from_numpy(
    semantic_embeddings_train.astype(np.float32)
)

X_text_val = torch.from_numpy(
    semantic_embeddings_val.astype(np.float32)
)

X_text_test = torch.from_numpy(
    semantic_embeddings_test.astype(np.float32)
)

X_image_train = torch.from_numpy(
    image_train.astype(np.float32)
)

X_image_val = torch.from_numpy(
    image_val.astype(np.float32)
)

X_image_test = torch.from_numpy(
    image_test.astype(np.float32)
)

temp_merge = pd.concat(
    [X_other_pre, adaptation_df],
    axis=1
)

X_other = torch.from_numpy(
    temp_merge.to_numpy(dtype=np.float32)
)

y_score = torch.tensor(
    df["wc_z"].to_numpy(dtype=np.float32)
)

In [65]:
other_train = X_other[train_idx]
other_val = X_other[val_idx]
other_test = X_other[test_idx]

score_train = y_score[train_idx]
score_val = y_score[val_idx]
score_test = y_score[test_idx]

In [66]:
train_dataset = AnimeDataset(
    X_text_train,
    X_image_train,
    other_train,
    score_train,
    studio_indices_train,
    studio_offsets_train,
    producer_indices_train,
    producer_offsets_train
)

val_dataset = AnimeDataset(
    X_text_val,
    X_image_val,
    other_val,
    score_val,
    studio_indices_val,
    studio_offsets_val,
    producer_indices_val,
    producer_offsets_val
)

test_dataset = AnimeDataset(
    X_text_test,
    X_image_test,
    other_test,
    score_test,
    studio_indices_test,
    studio_offsets_test,
    producer_indices_test,
    producer_offsets_test
)

In [67]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
)

In [68]:
full_dataset = ConcatDataset([train_dataset, val_dataset])
n_samples = len(full_dataset)

k = 5
kfold = KFold(n_splits=k, shuffle=True, random_state=42)

batch_size = train_loader.batch_size
collate_fn = train_loader.collate_fn 

criterion = nn.MSELoss()
patience = 15
n_epochs = 100

fold_results = []          
fold_model_states = []    

for fold, (train_idx, val_idx) in enumerate(kfold.split(np.arange(n_samples))):

    print(f"\n===== Fold {fold + 1}/{k} =====")

    model = FusionNetwork().to(device)  
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=2e-4,
        weight_decay=1e-2
    )

    train_subset = Subset(full_dataset, train_idx)
    val_subset = Subset(full_dataset, val_idx)

    fold_train_loader = DataLoader(
        train_subset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_fn
    )
    fold_val_loader = DataLoader(
        val_subset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn
    )

    best_val_loss = float("inf")
    patience_counter = 0
    best_model_state = None

    for epoch in range(n_epochs):

        model.train()
        train_loss = 0

        for (
            text, image, other,
            studio_indices, studio_offsets,
            producer_indices, producer_offsets,
            target
        ) in fold_train_loader:

            text = text.to(device)
            image = image.to(device)
            other = other.to(device)
            studio_indices = studio_indices.to(device)
            studio_offsets = studio_offsets.to(device)
            producer_indices = producer_indices.to(device)
            producer_offsets = producer_offsets.to(device)
            target = target.to(device)

            prediction = model(
                text, image, other,
                studio_indices, studio_offsets,
                producer_indices, producer_offsets
            )

            loss = criterion(prediction.squeeze(-1), target)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        avg_train_loss = train_loss / len(fold_train_loader)

        model.eval()
        val_loss = 0

        with torch.no_grad():
            for (
                text, image, other,
                studio_indices, studio_offsets,
                producer_indices, producer_offsets,
                target
            ) in fold_val_loader:

                text = text.to(device)
                image = image.to(device)
                other = other.to(device)
                studio_indices = studio_indices.to(device)
                studio_offsets = studio_offsets.to(device)
                producer_indices = producer_indices.to(device)
                producer_offsets = producer_offsets.to(device)
                target = target.to(device)

                prediction = model(
                    text, image, other,
                    studio_indices, studio_offsets,
                    producer_indices, producer_offsets
                )

                loss = criterion(prediction.squeeze(-1), target)
                val_loss += loss.item()

        avg_val_loss = val_loss / len(fold_val_loader)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            best_model_state = copy.deepcopy(model.state_dict())
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Fold {fold + 1} early stopping at epoch {epoch}.")
                break

        if epoch % 5 == 0 or patience_counter == 0:
            print(
                f"  Epoch {epoch}: "
                f"Train Loss: {avg_train_loss:.4f} | "
                f"Val Loss: {avg_val_loss:.4f}"
            )

    print(f"Fold {fold + 1} best val loss: {best_val_loss:.4f}")
    fold_results.append(best_val_loss)
    fold_model_states.append(best_model_state)

fold_results = np.array(fold_results)
print(f"\n===== CV Results ({k}-fold) =====")
print(f"Per-fold val loss: {fold_results}")
print(f"Mean: {fold_results.mean():.4f}  |  Std: {fold_results.std():.4f}")

best_fold_idx = fold_results.argmin()
best_model_state = fold_model_states[best_fold_idx]
model = FusionNetwork().to(device)
model.load_state_dict(best_model_state)
print(f"\nLoaded weights from fold {best_fold_idx + 1} (val loss {fold_results[best_fold_idx]:.4f})")


===== Fold 1/5 =====
  Epoch 0: Train Loss: 0.6809 | Val Loss: 0.4775
  Epoch 1: Train Loss: 0.4450 | Val Loss: 0.4207
  Epoch 2: Train Loss: 0.3961 | Val Loss: 0.4018
  Epoch 4: Train Loss: 0.3300 | Val Loss: 0.3890
  Epoch 5: Train Loss: 0.2938 | Val Loss: 0.3870
  Epoch 7: Train Loss: 0.2593 | Val Loss: 0.3866
  Epoch 8: Train Loss: 0.2314 | Val Loss: 0.3824
  Epoch 10: Train Loss: 0.2120 | Val Loss: 0.3790
  Epoch 15: Train Loss: 0.1610 | Val Loss: 0.3879
  Epoch 17: Train Loss: 0.1468 | Val Loss: 0.3787
  Epoch 20: Train Loss: 0.1338 | Val Loss: 0.3908
  Epoch 21: Train Loss: 0.1303 | Val Loss: 0.3740
  Epoch 25: Train Loss: 0.1210 | Val Loss: 0.3938
  Epoch 30: Train Loss: 0.1023 | Val Loss: 0.3805
  Epoch 35: Train Loss: 0.0979 | Val Loss: 0.3894
Fold 1 early stopping at epoch 36.
Fold 1 best val loss: 0.3740

===== Fold 2/5 =====
  Epoch 0: Train Loss: 0.7105 | Val Loss: 0.4691
  Epoch 1: Train Loss: 0.4512 | Val Loss: 0.3806
  Epoch 3: Train Loss: 0.3647 | Val Loss: 0.3665
  

In [69]:
model.eval()

total_squared_error = 0
total_samples = 0

all_predictions = []
all_targets = []

with torch.no_grad():

    for (
        text,
        image,
        other,
        studio_indices,
        studio_offsets,
        producer_indices,
        producer_offsets,
        target
    ) in test_loader:

        text = text.to(device)
        image = image.to(device)
        other = other.to(device)

        studio_indices = studio_indices.to(device)
        studio_offsets = studio_offsets.to(device)

        producer_indices = producer_indices.to(device)
        producer_offsets = producer_offsets.to(device)

        target = target.to(device)

        predictions = model(
            text,
            image,
            other,
            studio_indices,
            studio_offsets,
            producer_indices,
            producer_offsets
        ).squeeze(-1)

        squared_error = (
            (predictions - target) ** 2
        ).sum()

        total_squared_error += squared_error.item()
        total_samples += target.size(0)

        all_predictions.append(
            predictions.cpu()
        )

        all_targets.append(
            target.cpu()
        )

final_mse = (
    total_squared_error /
    total_samples
)

final_rmse = np.sqrt(final_mse)

final_predictions = torch.cat(
    all_predictions
).numpy()

final_targets = torch.cat(
    all_targets
).numpy()

print(f"Test MSE:  {final_mse:.4f}")
print(f"Test RMSE: {final_rmse:.4f}")

Test MSE:  0.3895
Test RMSE: 0.6241


### Drop Rate Prediction

In [21]:
model = FusionNetwork()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

X_text_train = torch.from_numpy(
    semantic_embeddings_train.astype(np.float32)
)

X_text_val = torch.from_numpy(
    semantic_embeddings_val.astype(np.float32)
)

X_text_test = torch.from_numpy(
    semantic_embeddings_test.astype(np.float32)
)

X_image_train = torch.from_numpy(
    image_train.astype(np.float32)
)

X_image_val = torch.from_numpy(
    image_val.astype(np.float32)
)

X_image_test = torch.from_numpy(
    image_test.astype(np.float32)
)

temp_merge = pd.concat(
    [X_other_pre, adaptation_df],
    axis=1
)

X_other = torch.from_numpy(
    temp_merge.to_numpy(dtype=np.float32)
)

y_score = torch.tensor(
    df["drop_rate_z"].to_numpy(dtype=np.float32)
)

In [22]:
other_train = X_other[train_idx]
other_val = X_other[val_idx]
other_test = X_other[test_idx]

score_train = y_score[train_idx]
score_val = y_score[val_idx]
score_test = y_score[test_idx]

In [28]:
train_dataset = AnimeDataset(
    X_text_train,
    X_image_train,
    other_train,
    score_train,
    studio_indices_train,
    studio_offsets_train,
    producer_indices_train,
    producer_offsets_train
)

val_dataset = AnimeDataset(
    X_text_val,
    X_image_val,
    other_val,
    score_val,
    studio_indices_val,
    studio_offsets_val,
    producer_indices_val,
    producer_offsets_val
)

test_dataset = AnimeDataset(
    X_text_test,
    X_image_test,
    other_test,
    score_test,
    studio_indices_test,
    studio_offsets_test,
    producer_indices_test,
    producer_offsets_test
)

In [29]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
)

In [30]:
full_dataset = ConcatDataset([train_dataset, val_dataset])
n_samples = len(full_dataset)

k = 5
kfold = KFold(n_splits=k, shuffle=True, random_state=42)

batch_size = train_loader.batch_size
collate_fn = train_loader.collate_fn 

criterion = nn.MSELoss()
patience = 15
n_epochs = 100

fold_results = []          
fold_model_states = []    

for fold, (train_idx, val_idx) in enumerate(kfold.split(np.arange(n_samples))):

    print(f"\n===== Fold {fold + 1}/{k} =====")

    model = FusionNetwork().to(device)  
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=2e-4,
        weight_decay=1e-2
    )

    train_subset = Subset(full_dataset, train_idx)
    val_subset = Subset(full_dataset, val_idx)

    fold_train_loader = DataLoader(
        train_subset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=collate_fn
    )
    fold_val_loader = DataLoader(
        val_subset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_fn
    )

    best_val_loss = float("inf")
    patience_counter = 0
    best_model_state = None

    for epoch in range(n_epochs):

        model.train()
        train_loss = 0

        for (
            text, image, other,
            studio_indices, studio_offsets,
            producer_indices, producer_offsets,
            target
        ) in fold_train_loader:

            text = text.to(device)
            image = image.to(device)
            other = other.to(device)
            studio_indices = studio_indices.to(device)
            studio_offsets = studio_offsets.to(device)
            producer_indices = producer_indices.to(device)
            producer_offsets = producer_offsets.to(device)
            target = target.to(device)

            prediction = model(
                text, image, other,
                studio_indices, studio_offsets,
                producer_indices, producer_offsets
            )

            loss = criterion(prediction.squeeze(-1), target)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        avg_train_loss = train_loss / len(fold_train_loader)

        model.eval()
        val_loss = 0

        with torch.no_grad():
            for (
                text, image, other,
                studio_indices, studio_offsets,
                producer_indices, producer_offsets,
                target
            ) in fold_val_loader:

                text = text.to(device)
                image = image.to(device)
                other = other.to(device)
                studio_indices = studio_indices.to(device)
                studio_offsets = studio_offsets.to(device)
                producer_indices = producer_indices.to(device)
                producer_offsets = producer_offsets.to(device)
                target = target.to(device)

                prediction = model(
                    text, image, other,
                    studio_indices, studio_offsets,
                    producer_indices, producer_offsets
                )

                loss = criterion(prediction.squeeze(-1), target)
                val_loss += loss.item()

        avg_val_loss = val_loss / len(fold_val_loader)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            best_model_state = copy.deepcopy(model.state_dict())
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Fold {fold + 1} early stopping at epoch {epoch}.")
                break

        if epoch % 5 == 0 or patience_counter == 0:
            print(
                f"  Epoch {epoch}: "
                f"Train Loss: {avg_train_loss:.4f} | "
                f"Val Loss: {avg_val_loss:.4f}"
            )

    print(f"Fold {fold + 1} best val loss: {best_val_loss:.4f}")
    fold_results.append(best_val_loss)
    fold_model_states.append(best_model_state)

fold_results = np.array(fold_results)
print(f"\n===== CV Results ({k}-fold) =====")
print(f"Per-fold val loss: {fold_results}")
print(f"Mean: {fold_results.mean():.4f}  |  Std: {fold_results.std():.4f}")

best_fold_idx = fold_results.argmin()
best_model_state = fold_model_states[best_fold_idx]
model = FusionNetwork().to(device)
model.load_state_dict(best_model_state)
print(f"\nLoaded weights from fold {best_fold_idx + 1} (val loss {fold_results[best_fold_idx]:.4f})")


===== Fold 1/5 =====
  Epoch 0: Train Loss: 0.8649 | Val Loss: 0.6457
  Epoch 1: Train Loss: 0.7666 | Val Loss: 0.5981
  Epoch 5: Train Loss: 0.4022 | Val Loss: 0.6135
  Epoch 6: Train Loss: 0.3591 | Val Loss: 0.5978
  Epoch 7: Train Loss: 0.3178 | Val Loss: 0.5910
  Epoch 8: Train Loss: 0.2812 | Val Loss: 0.5902
  Epoch 9: Train Loss: 0.2462 | Val Loss: 0.5850
  Epoch 10: Train Loss: 0.2397 | Val Loss: 0.5936
  Epoch 15: Train Loss: 0.1840 | Val Loss: 0.5938
  Epoch 19: Train Loss: 0.1609 | Val Loss: 0.5844
  Epoch 20: Train Loss: 0.1568 | Val Loss: 0.5883
  Epoch 21: Train Loss: 0.1619 | Val Loss: 0.5738
  Epoch 25: Train Loss: 0.1394 | Val Loss: 0.5812
  Epoch 26: Train Loss: 0.1477 | Val Loss: 0.5618
  Epoch 30: Train Loss: 0.1308 | Val Loss: 0.5810
  Epoch 35: Train Loss: 0.1218 | Val Loss: 0.5868
  Epoch 40: Train Loss: 0.1094 | Val Loss: 0.5746
Fold 1 early stopping at epoch 41.
Fold 1 best val loss: 0.5618

===== Fold 2/5 =====
  Epoch 0: Train Loss: 0.8468 | Val Loss: 0.7127


In [31]:
model.eval()

total_squared_error = 0
total_samples = 0

all_predictions = []
all_targets = []

with torch.no_grad():

    for (
        text,
        image,
        other,
        studio_indices,
        studio_offsets,
        producer_indices,
        producer_offsets,
        target
    ) in test_loader:

        text = text.to(device)
        image = image.to(device)
        other = other.to(device)

        studio_indices = studio_indices.to(device)
        studio_offsets = studio_offsets.to(device)

        producer_indices = producer_indices.to(device)
        producer_offsets = producer_offsets.to(device)

        target = target.to(device)

        predictions = model(
            text,
            image,
            other,
            studio_indices,
            studio_offsets,
            producer_indices,
            producer_offsets
        ).squeeze(-1)

        squared_error = (
            (predictions - target) ** 2
        ).sum()

        total_squared_error += squared_error.item()
        total_samples += target.size(0)

        all_predictions.append(
            predictions.cpu()
        )

        all_targets.append(
            target.cpu()
        )

final_mse = (
    total_squared_error /
    total_samples
)

final_rmse = np.sqrt(final_mse)

final_predictions = torch.cat(
    all_predictions
).numpy()

final_targets = torch.cat(
    all_targets
).numpy()

print(f"Test MSE:  {final_mse:.4f}")
print(f"Test RMSE: {final_rmse:.4f}")

Test MSE:  0.7298
Test RMSE: 0.8543
